# Institutional Accumulation — **monthly** backtest (NSE)

Python port of the Pine **Institutional Accumulation Strategy** (no lookahead). Same default thresholds as TradingView:

- Money flow / relative volume / OBV / ADL score (0–5), enter when score ≥ 4
- 12-month trend EMA filter, 10% stop, signal exits, 3-month cooldown
- 10% of equity per new long, next **month open** fills (`process_orders_on_close=false`)

Outputs: `$QUANT_DATA_DIR/results/institutional_accumulation_monthly/`

**Resume:** each phase writes parquet + `meta/*_meta.json` fingerprints. Skip rebuild unless `CFG["rerun_all"] = True` or `CFG["force_rerun"]` lists the phase.

Prerequisite: daily `ohlcv_panel.parquet` (run **explosive_moves_research** Phase 2 if missing).

In [ ]:
from pathlib import Path
import json, sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "quant").is_dir():
    REPO_ROOT = REPO_ROOT.parent
for p in (REPO_ROOT / "src", REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

for pkg in ("pandas", "pyarrow", "numpy", "tqdm"):
    root = pkg.split(".")[0].replace("-", "_")
    if importlib.util.find_spec(root) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from quant.config import DATA_DIR
from quant.research.explosive_research import (
    fingerprint_dict,
    force_rerun_phase,
    phase_cache_hit,
    write_phase_meta,
    read_meta_fingerprint,
    file_fingerprint,
)
from quant.research.institutional_accumulation import (
    DEFAULT_PARAMS,
    INST_ACC_ENGINE_VERSION,
    strategy_params_fingerprint,
    load_daily_ohlcv_panel,
    resample_panel_to_monthly,
    compute_signals_monthly,
    backtest_portfolio_monthly,
    backtest_symbol_monthly,
    summarize_backtest,
)

OUT_DIR = DATA_DIR / "results" / "institutional_accumulation_monthly"
META_DIR = OUT_DIR / "meta"
for sub in tqdm(("panel", "monthly", "signals", "backtest", META_DIR), desc="Setup — output dirs"):
    (OUT_DIR / sub if sub != META_DIR else META_DIR).mkdir(parents=True, exist_ok=True)

DAILY_PANEL_PATH = OUT_DIR / "panel" / "daily_ohlcv_filtered.parquet"
MONTHLY_PATH = OUT_DIR / "monthly" / "ohlcv_monthly.parquet"
SIGNALS_PATH = OUT_DIR / "signals" / "monthly_signals.parquet"
TRADES_PATH = OUT_DIR / "backtest" / "portfolio_trades.parquet"
EQUITY_PATH = OUT_DIR / "backtest" / "portfolio_equity.parquet"
SYMBOL_STATS_PATH = OUT_DIR / "backtest" / "per_symbol_summary.parquet"

DAILY_META = META_DIR / "daily_meta.json"
MONTHLY_META = META_DIR / "monthly_meta.json"
SIGNALS_META = META_DIR / "signals_meta.json"
BACKTEST_META = META_DIR / "backtest_meta.json"

pd.set_option("display.width", 200)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

## Config

In [ ]:
with tqdm(total=1, desc="Phase 0 — config") as pbar:
    CFG = {
        "rerun_all": False,
        "force_rerun": [],
        "date_start": "2005-01-01",
        "date_end": None,
        "spotlight_symbol": None,
        "run_per_symbol_pass": True,
        **strategy_params_fingerprint(DEFAULT_PARAMS),
    }
    PARAMS = {k: CFG[k] for k in DEFAULT_PARAMS if k in CFG}
    pbar.update(1)

print("Engine:", INST_ACC_ENGINE_VERSION)
print("Strategy params:", {k: PARAMS[k] for k in (
    "required_score", "stop_loss_percent", "cooldown_months", "position_pct", "initial_capital"
)})

## Phase 1 — daily panel (cached)

In [ ]:
_daily_fp = fingerprint_dict({
    "engine": INST_ACC_ENGINE_VERSION,
    "date_start": CFG["date_start"],
    "date_end": CFG["date_end"],
    "eq_only": PARAMS["eq_only"],
    "source": file_fingerprint(DATA_DIR / "results" / "explosive_moves" / "panel" / "ohlcv_panel.parquet"),
})

if phase_cache_hit(CFG, "daily", [DAILY_PANEL_PATH], DAILY_META, _daily_fp):
    daily = pd.read_parquet(DAILY_PANEL_PATH)
else:
    with tqdm(total=2, desc="Phase 1 — daily panel") as bar:
        daily = load_daily_ohlcv_panel(
            date_start=CFG["date_start"],
            date_end=CFG["date_end"],
            eq_only=PARAMS["eq_only"],
        )
        bar.update(1)
        daily.to_parquet(DAILY_PANEL_PATH, index=False)
        write_phase_meta(DAILY_META, "daily", _daily_fp, n_rows=len(daily))
        bar.update(1)

daily["Date"] = pd.to_datetime(daily["Date"])
print("Daily:", daily.shape, daily["Date"].min(), "→", daily["Date"].max(), "| symbols:", daily["Symbol"].nunique())

## Phase 2 — resample to calendar months (cached)

In [ ]:
_monthly_fp = fingerprint_dict({
    "daily": read_meta_fingerprint(DAILY_META),
    "engine": INST_ACC_ENGINE_VERSION,
})

if phase_cache_hit(CFG, "monthly", [MONTHLY_PATH], MONTHLY_META, _monthly_fp):
    monthly = pd.read_parquet(MONTHLY_PATH)
else:
    parts = []
    syms = daily["Symbol"].unique()
    for sym in tqdm(syms, desc="Phase 2 — monthly bars", unit="symbol"):
        g = daily.loc[daily["Symbol"] == sym, ["Date", "Open", "High", "Low", "Close", "Volume"]]
        from quant.research.institutional_accumulation import resample_symbol_to_monthly
        m = resample_symbol_to_monthly(g)
        if m.empty:
            continue
        m["Symbol"] = sym
        parts.append(m)
    monthly = pd.concat(parts, ignore_index=True)
    monthly.to_parquet(MONTHLY_PATH, index=False)
    write_phase_meta(MONTHLY_META, "monthly", _monthly_fp, n_symbols=monthly["Symbol"].nunique())

monthly["Date"] = pd.to_datetime(monthly["Date"])
print("Monthly:", monthly.shape, "| symbols:", monthly["Symbol"].nunique())

## Phase 3 — accumulation signals (cached)

In [ ]:
_signals_fp = fingerprint_dict({
    "monthly": read_meta_fingerprint(MONTHLY_META),
    "params": PARAMS,
    "engine": INST_ACC_ENGINE_VERSION,
})

if phase_cache_hit(CFG, "signals", [SIGNALS_PATH], SIGNALS_META, _signals_fp):
    signals = pd.read_parquet(SIGNALS_PATH)
else:
    signals = compute_signals_monthly(monthly, PARAMS)
    signals.to_parquet(SIGNALS_PATH, index=False)
    n_buy = int(signals["buy_signal"].sum()) if not signals.empty else 0
    write_phase_meta(SIGNALS_META, "signals", _signals_fp, n_buy=n_buy)

if signals.empty:
    raise RuntimeError("No monthly signals — check data range / min_monthly_bars.")
print("Signal rows:", len(signals), "| buy flags:", int(signals["buy_signal"].sum()))
display(signals.loc[signals["buy_signal"], ["Date", "Symbol", "accumulation_score", "money_flow", "relative_volume"]].head(15))

## Phase 4 — portfolio backtest (cached)

In [ ]:
_bt_fp = fingerprint_dict({
    "signals": read_meta_fingerprint(SIGNALS_META),
    "params": PARAMS,
    "engine": INST_ACC_ENGINE_VERSION,
})

if phase_cache_hit(CFG, "backtest", [TRADES_PATH, EQUITY_PATH], BACKTEST_META, _bt_fp):
    trades = pd.read_parquet(TRADES_PATH)
    equity = pd.read_parquet(EQUITY_PATH)
    symbol_stats = pd.read_parquet(SYMBOL_STATS_PATH) if SYMBOL_STATS_PATH.exists() else pd.DataFrame()
else:
    steps = 2 + (1 if CFG.get("run_per_symbol_pass") else 0)
    with tqdm(total=steps, desc="Phase 4 — backtest") as bar:
        trades, equity = backtest_portfolio_monthly(signals, PARAMS)
        bar.update(1)
        trades.to_parquet(TRADES_PATH, index=False)
        equity.to_parquet(EQUITY_PATH, index=False)
        bar.update(1)
        symbol_stats = pd.DataFrame()
        if CFG.get("run_per_symbol_pass"):
            rows = []
            for sym, g in tqdm(
                monthly.groupby("Symbol", observed=True),
                desc="Per-symbol backtest",
                leave=False,
            ):
                if len(g) < PARAMS["min_monthly_bars"]:
                    continue
                t, e = backtest_symbol_monthly(g, PARAMS)
                s = summarize_backtest(t, e, PARAMS["initial_capital"])
                rows.append({"Symbol": sym, **s.__dict__})
            symbol_stats = pd.DataFrame(rows)
            symbol_stats.to_parquet(SYMBOL_STATS_PATH, index=False)
            bar.update(1)
        write_phase_meta(BACKTEST_META, "backtest", _bt_fp, n_trades=len(trades))

portfolio_summary = summarize_backtest(trades, equity, PARAMS["initial_capital"])
print("Portfolio profitable:", portfolio_summary.profitable)
print(portfolio_summary)

## Phase 5 — profitability report (fast; always runs)

In [ ]:
report = {
    "initial_capital": portfolio_summary.initial_capital,
    "final_equity": portfolio_summary.final_equity,
    "total_return_pct": round(portfolio_summary.total_return_pct, 2),
    "cagr_pct": round(portfolio_summary.cagr_pct, 2),
    "max_drawdown_pct": round(portfolio_summary.max_drawdown_pct, 2),
    "n_trades": portfolio_summary.n_trades,
    "win_rate_pct": round(portfolio_summary.win_rate_pct, 2),
    "profitable": portfolio_summary.profitable,
}
display(pd.DataFrame([report]))

if not equity.empty:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(pd.to_datetime(equity["Date"]), equity["equity"], label="Portfolio equity")
        ax.axhline(PARAMS["initial_capital"], color="gray", ls="--", lw=1)
        ax.set_title("Institutional accumulation — monthly portfolio equity")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib not installed — skipping chart")

if "symbol_stats" in dir() and not symbol_stats.empty:
    pct_profitable = 100 * symbol_stats["profitable"].mean()
    print(f"Symbols with isolated backtest profit: {pct_profitable:.1f}%")
    display(
        symbol_stats.sort_values("total_return_pct", ascending=False)[
            ["Symbol", "total_return_pct", "cagr_pct", "max_drawdown_pct", "n_trades", "win_rate_pct"]
        ].head(20)
    )

spot = CFG.get("spotlight_symbol")
if spot:
    g = monthly.loc[monthly["Symbol"] == spot.upper()]
    if not g.empty:
        t, e = backtest_symbol_monthly(g, PARAMS)
        print("Spotlight", spot, summarize_backtest(t, e, PARAMS["initial_capital"]))
        display(t.tail(10))

summary_path = OUT_DIR / "backtest" / "profitability_summary.json"
summary_path.write_text(json.dumps(report, indent=2))
print("Wrote", summary_path)